# HPO.ipynb — Optimal hyperparameters check & update

**Goal:** verify whether the hyperparameters in `Experiment.py` match the tuned “best” settings from your Excel tuning files, then apply updates where needed.

## What I used as the source of “optimal”
- DQN / REINFORCE: `data sheets/tuned hyperparameters/*.xlsx`
- AC / A2C / PPO / SAC: `data sheets/higher terminal steps/*.xlsx` and `data sheets/temp2/*.xlsx`

## Change summary
- ✅ All algorithms match tuned values **except one**.
- **UPDATED:** `REINFORCE.actor_hidden_nn` from `[[32, 32]]` → `[[128, 128]]`.


## Global training/evaluation settings (from `global_config` in `Experiment.py`)

These are shared across algorithms and are consistent with the tuning workbooks you have (notably: `n_timesteps=200000`, `max_train_episode_length=1000`, `max_eval_episode_length=20000`, and `eval_with_env_episode_trials=True`).

| Hyperparameter | Value | Short explanation |
|---|---:|---|
| `n_repetitions` | 5 | Number of independent training runs (used to compute mean/std curves). |
| `UNUSED_CPU_CORES` | 3 | Leaves CPU headroom so parallel workers don’t starve other processes. |
| `benchmark_curve` | 1 | Which baseline CSV to plot for comparison. |
| `plot_smoothing_window` | [1, 101, 201, 251, 351] | Savitzky–Golay smoothing windows for plotting. |
| `curve_confidence_interval` | 0.6 | Confidence interval shading factor (t-distribution based). |
| `curve_shaded_area_opacity` | 0.06 | Opacity of the shaded confidence band. |
| `show_curve_plots` | True | Whether to display the learning curve figures. |
| `show_curved_plots` | True | Whether to display the smoothed curve window at the end. |
| `animation_plot` | True | Whether to show CartPole animation. |
| `use_existing_disk_data` | True | Reuse Excel results when matching hyperparameter signatures exist. |
| `use_existing_disk_trained_networks` | True | Reuse stored checkpoints when matching architecture signatures exist. |
| `max_train_episode_length` | 1000 | Episode truncation length during training. |
| `base_seed` | 42 | Base RNG seed; each repetition uses a derived seed. |
| `n_timesteps` | 200000 | Total environment steps per run. |
| `max_eval_episode_length` | 20000 | Episode truncation length during evaluation trials. |
| `eval_interval` | 250 | Evaluate/log every N training steps. |
| `eval_with_env_episode_trials` | True | Uses greedy **environment** evaluation at each interval (slower, more faithful). |
| `n_eval_episodes` | 5 | Number of evaluation episodes averaged per evaluation point. |


## DQN (optimal tuned values)

| Hyperparameter | Optimal value (used) | Short explanation |
|---|---:|---|
| `gamma` | 0.99 | Discount factor for future rewards. |
| `learning_rate` | 1e-3 | Adam learning rate for Q-network. |
| `nn_hidden_layer_widths` | [[128, 128]] | Two-layer MLP width for Q approximator. |
| `exploration_method` | "egreedy" | Uses ε-greedy exploration (vs Boltzmann softmax). |
| `epsilons` | [0.05] | Candidate ε values for fixed-ε sweeps (here only 0.05). |
| `epsilon_start` | 0.05 | Initial ε for ε-greedy. |
| `epsilon_end` | 0.05 | Final ε (here equal to start, effectively constant). |
| `epsilon_decay` | 1 | ε decay multiplier; with decay disabled (next key), it doesn’t matter. |
| `epsilon_decay_interval` | 0 | **0 disables ε-decay trials** (keeps ε constant). |
| `softmax_temps` | [1.0, 0.5, 0.1] | Candidate temperatures for softmax exploration (not used because `egreedy`). |
| `TN_active` | True | Enables Target Network to stabilize bootstrap targets. |
| `TN_step` | 100 | Target network update interval (steps). |
| `ER_active` | True | Enables Experience Replay to reduce correlation. |
| `ER_replay_buffer_size` | 80000 | Replay buffer capacity. |
| `ER_batch_size` | [64] | Mini-batch size sampled from replay buffer. |
| `ER_min_replay_size` | 2000 | Minimum buffer size before training starts. |
| `ER_sample_train_frequency` | [1] | How often to train relative to environment steps. |
| `ER_replay_ratio` | 1.0 | Number of replay updates per scheduled training step. |


## REINFORCE (optimal tuned values)

| Hyperparameter | Optimal value (used) | Short explanation | Status |
|---|---:|---|---|
| `gamma` | 0.99 | Discount factor for returns. | |
| `actor_lr` | 1e-3 | Policy-gradient learning rate for the actor. | |
| `actor_hidden_nn` | [[128, 128]] | Actor MLP architecture (REINFORCE policy network). | **✅ CHANGED** |

### Note on the change
- You previously had `[[32, 32]]`.
- The tuned best setting found in the Excel workbook uses `[[128, 128]]`.


## AC (optimal tuned values)

| Hyperparameter | Optimal value (used) | Short explanation |
|---|---:|---|
| `gamma` | 0.99 | Discount factor. |
| `actor_lr` | 1e-3 | Actor learning rate. |
| `actor_hidden_nn` | [[64, 64]] | Actor network width. |
| `critic_lr` | 1e-3 | Critic learning rate. |
| `critic_hidden_nn` | [[128, 128]] | Critic network width. |
| `TN_step` | 10 | N-step return horizon used in the AC/TD target construction. |

## A2C (optimal tuned values)

| Hyperparameter | Optimal value (used) | Short explanation |
|---|---:|---|
| `gamma` | 0.99 | Discount factor. |
| `actor_lr` | 1e-4 | Actor/policy learning rate. |
| `actor_hidden_nn` | [[64, 64]] | Actor network width. |
| `critic_lr` | 0.01 | Critic/value-function learning rate. |
| `critic_hidden_nn` | [[128, 128]] | Critic network width. |
| `TN_step` | 10 | n-step return (bootstrap horizon) used for the critic target. |


## PPO (optimal tuned values)

| Hyperparameter | Optimal value (used) | Short explanation |
|---|---:|---|
| `gamma` | 0.99 | Discount factor. |
| `actor_lr` | 3e-4 | Actor learning rate. |
| `actor_hidden_nn` | [[64, 64]] | Actor MLP width. |
| `critic_lr` | 1e-3 | Critic/value learning rate. |
| `critic_hidden_nn` | [[64, 64]] | Critic MLP width. |
| `gae_lambda` | 0.95 | GAE bias/variance tradeoff parameter. |
| `clip_eps` | 0.2 | PPO clipping range for the surrogate objective. |
| `n_epochs` | 10 | Number of optimization epochs per rollout. |
| `rollout_steps` | 2048 | Buffer length of collected rollout before PPO updates. |
| `mini_batch_size` | 64 | Mini-batch size for each PPO epoch. |
| `entropy_coef` | 0.0 | Entropy bonus weight (0 disables entropy regularization). |
| `value_coef` | 0.5 | Weight of value loss in total loss. |
| `max_grad_norm` | 0.5 | Gradient norm clipping to keep updates stable. |


## SAC (optimal tuned values)

| Hyperparameter | Optimal value (used) | Short explanation |
|---|---:|---|
| `gamma` | 0.99 | Discount factor. |
| `actor_lr` | 3e-4 | Actor learning rate. |
| `actor_hidden_nn` | [[64, 64]] | Actor MLP width. |
| `critic_lr` | 3e-4 | Q-network learning rate (for both critics). |
| `critic_hidden_nn` | [[128, 128]] | Q-network width (twin Qs). |
| `alpha_lr` | 3e-4 | Learning rate for entropy temperature (when auto-tuning). |
| `tau` | 5e-3 | Polyak averaging rate for target networks. |
| `target_entropy_ratio` | 0.98 | Target entropy expressed as a ratio of `log(n_actions)`. |
| `replay_buffer_size` | 1e5 | Experience replay capacity. |
| `batch_size` | 64 | SGD mini-batch size from replay buffer. |
| `warmup_steps` | 1e3 | Number of random steps before learning begins. |
| `updates_per_step` | 1 | Number of gradient update steps per environment step. |
| `auto_tune_alpha` | True | Enables automatic entropy-temperature tuning. |
| `alpha_init` | 1.0 | Initial entropy temperature (seed for auto-tuning). |


## Final verification result
- ✅ DQN matches tuned best values.
- ✅ AC matches tuned best values.
- ✅ A2C matches tuned best values.
- ✅ PPO matches tuned best values.
- ✅ SAC matches tuned best values.
- ✅ REINFORCE updated to tuned best value (**`actor_hidden_nn=[[128,128]]`**).
